In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
data= pd.read_csv("exactkm.csv")


# feature engineering

# Change KM_range values to the interval mean, except for +80000 values
def calcular_media(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip()

    if '+' in x:
        return 80000

    minimo, maximo = map(int, x.split('-'))
    return (minimo + maximo) / 2

data['KM_Range_Before_Defect'] = data['KM_Range_Before_Defect'].apply(calcular_media)


# feature engineering for cathegorical features, we have 4 ordinal cathegories and 2 nominal cathegories, therefore they are gonna be treated in a different way

order_tire_condition = ['Worn', 'Critical', 'Moderate', 'Good']
order_vibration_level = ['Medium', 'Low', 'High']

ordinal_columns = ["Tire_Condition", "Vibration_Level"]
transform_processing = ColumnTransformer(transformers=[("ordinal", OrdinalEncoder(categories=[order_tire_condition, order_vibration_level]),
                                                        ordinal_columns)], remainder="passthrough").set_output(transform="pandas")
data = transform_processing.fit_transform(data)


nominal_columns = ["remainder__Vehicle_Type","remainder__Accident_History"]
transform_processing = ColumnTransformer(transformers=[("nominal", OneHotEncoder(handle_unknown="ignore",sparse_output=False),
                                                        nominal_columns)], remainder="passthrough").set_output(transform="pandas")


data = transform_processing.fit_transform(data)



# Train_test split for training
Y_train = data["remainder__remainder__Remaining_KM"].to_numpy().reshape(-1, 1)
X_train = data.drop("remainder__remainder__Remaining_KM", axis=1).to_numpy()



#Y_train is an 1D array, for this reason we need to reshape it

x_train, x_test, y_train, y_test = train_test_split(X_train, Y_train, train_size=0.8, test_size=0.2)

x_test, x_valid, y_test, y_valid = train_test_split(x_test,y_test, train_size=0.5, test_size=0.5)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_valid = scaler.transform(x_valid)
import keras

model = keras.models.Sequential()
model.add(keras.layers.Dense(300, activation=keras.activations.relu, kernel_initializer=keras.initializers.he_normal))
model.add(keras.layers.Dense(300, activation=keras.activations.relu, kernel_initializer=keras.initializers.he_normal))
model.add(keras.layers.Dense(1))

model.compile(loss="mse", optimizer="Adam", metrics=["mae"])

hist = model.fit(x_train, y_train, epochs=50, validation_data=(x_valid, y_valid))


Epoch 1/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3222946560.0000 - mae: 49016.3633 - val_loss: 1676826624.0000 - val_mae: 34919.3516
Epoch 2/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 566698624.0000 - mae: 18089.7422 - val_loss: 212679632.0000 - val_mae: 11445.9600
Epoch 3/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 180076656.0000 - mae: 10688.5586 - val_loss: 147237568.0000 - val_mae: 9830.5928
Epoch 4/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 132268304.0000 - mae: 9290.1816 - val_loss: 110862032.0000 - val_mae: 8583.1328
Epoch 5/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100908448.0000 - mae: 8163.0015 - val_loss: 85436360.0000 - val_mae: 7544.5034
Epoch 6/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 80519888.0000 - mae: 7338.4868 - val_loss: 70433880.0000 - val_mae: 6873.6079
Epoch 7/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 69028632.0000 - mae: 6837.6011 - val_loss: 62561396.0000 - val_mae: 6476.4951
Epoch 8/50
300/300